# Lecția 13 - Memoria agentului


## Configurare

Acest caiet demonstrează cum să construiți un agent de rezervare pentru călătorii cu **memorie persistentă** folosind **Microsoft Agent Framework** (MAF).

Veți învăța cum diferitele tipuri de memorie ale agentului — de lucru, pe termen scurt și pe termen lung — influențează modul în care un agent reține și folosește informațiile în cadrul conversațiilor.

**Prerechizite:**
- Un proiect Azure AI Foundry cu un model de chat implementat (ex. `gpt-4o-mini`).
- Conectat în Azure CLI — rulați `az login` în terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — endpointul proiectului dvs. Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — numele modelului dvs. implementat.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Tipuri de memorie ale agentului

Agenții AI pot folosi diferite tipuri de memorie, fiecare având un scop distinct:

### Memorie de lucru
Firul conversației în sine — mesajele schimbate într-o singură sesiune. Agentul poate face referire la mesajele anterioare din același fir pentru a menține coerența. În MAF, aceasta este creată prin **`agent.create_session()`**, care returnează un `AgentSession`.

### Memorie pe termen scurt
Informații care persistă pe durata unei sarcini sau sesiuni, dar nu sunt stocate permanent. De exemplu, agentul poate acumula fapte în timpul unei conversații multi-turn de planificare și le poate folosi pentru a produce un itinerar final.

### Memorie pe termen lung
Preferințe și fapte care persistă **peste sesiuni**. Un utilizator care revine nu ar trebui să-și repete restricțiile alimentare sau stilul de călătorie. Memoria pe termen lung este de obicei susținută de un depozit extern — o bază de date, fișier sau index vectorial — și este accesată de agent prin intermediul uneltelor.


## Memorie de lucru cu sesiuni

Cea mai simplă formă de memorie este sesiunea de conversație. Când transmiți aceeași sesiune (creată prin `agent.create_session()`) la apeluri succesive `agent.run()`, agentul vede întreaga istorie a conversației și poate reaminti detalii anterioare.

Să creăm un agent de călătorii și să demonstrăm memoria de lucru.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agentul și-a amintit corect bugetul deoarece ambele mesaje împărtășesc aceeași sesiune. Aceasta este **memorie de lucru** — există doar pe durata sesiunii.

### Ce se întâmplă cu un fir nou?

Dacă creăm o sesiune **nouă**, agentul nu are memorie a conversației anterioare:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pattern pentru Memorie pe Termen Lung

Pentru a reține preferințele utilizatorului **pe parcursul sesiunilor**, avem nevoie de un depozit persistent care să existe în afara firului conversației. Agentul accesează acest depozit prin **unelte** — funcții pe care le poate apela pentru a salva și recupera informații.

Mai jos implementăm un depozit simplu de preferințe în memorie (în producție îl veți susține cu o bază de date sau un index vectorial) și îl expunem ca unelte pe care agentul le poate folosi.

### Arhitectură
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenariul 1 — Utilizatorul pentru prima dată rezervă o călătorie aniversară

Sarah vizitează pentru prima dată. Agentul ar trebui să salveze preferințele ei prin intermediul uneltelor și să le folosească pentru a recomanda hoteluri.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenariul 2 — Sarah revine după câteva săptămâni

Sarah începe un **fir complet nou** (simulând o sesiune nouă). Memoria de lucru este goală, dar depozitul de preferințe pe termen lung încă păstrează informațiile ei. Agentul ar trebui să le recupereze și să le folosească pentru a personaliza recomandările.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Rezumat

În această lecție ați învățat trei tipuri de memorie ale agentului și cum să le implementați cu Microsoft Agent Framework:

| Tip Memorie | Mecanism MAF | Durată de viață |
|---|---|---|
| **De lucru** | `agent.create_session()` | O singură conversație |
| **Pe termen scurt** | Context acumulat într-un fir de discuție | O singură sarcină / sesiune |
| **Pe termen lung** | Magazin extern accesat prin funcțiile `@tool` | Între sesiuni |

### Aspecte cheie
1. **`agent.create_session()`** oferă memorie de lucru — agentul vede toată istoricul conversației dintr-o sesiune.
2. **Sesiunile noi pierd contextul** — fără memorie pe termen lung agentul nu poate reaminti conversațiile anterioare.
3. **Funcțiile `@tool`** fac legătura — permit agentului să salveze și să recupereze informații dintr-un magazin persistent.
4. **Personalizarea se îmbunătățește în timp** — cu cât se stochează mai multe preferințe, cu atât recomandările agentului sunt mai bune.

### Aplicații în lumea reală
- **Serviciu clienți**: amintirea istoricului și preferințelor clienților
- **Asistenți personali**: menținerea contextului pe zile sau săptămâni
- **Îngrijire medicală**: urmărirea informațiilor și preferințelor pacienților
- **Comerț electronic**: cumpărături personalizate bazate pe istoric

### Pași următori
- Înlocuiți dicționarul din memorie cu o bază de date sau un magazin vectorial (de ex. Azure AI Search)
- Adăugați expirarea memoriei pentru informații sensibile în timp
- Construiți sisteme multi-agent cu memorie partajată
- Explorați caietul Cognee pentru memorie susținută de grafuri de cunoștințe


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare de responsabilitate**:
Acest document a fost tradus folosind serviciul de traducere automată AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim să oferim acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un specialist uman. Nu ne asumăm responsabilitatea pentru orice neînțelegeri sau interpretări greșite rezultate din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
